# 03 — OE2: Determinantes de los retornos (panel FE) y global vs local

Contrasta **H1–H4** (signo y magnitud de cada factor) y **H6** (dominancia global vs local).

Estrategia (Frente 3):
1. **Baseline** OLS sobre la cartera del sector con errores HAC (Newey-West).
2. **Principal** Panel FE con errores **Driscoll-Kraay** (corrige dependencia de sección cruzada).
3. Diagnósticos: Hausman (FE vs RE), CD-Pesaran (dependencia cruzada), VIF (multicolinealidad).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import config as C
from src import panel_models as pm
from src import transformations as T

## 1. Cargar el panel
El panel se arma con `T.construir_panel(retornos, macro, vars_empresa)`.
Aquí se asume guardado como `data/processed/panel.parquet` (MultiIndex empresa, fecha).

Cada variable macro I(1) debe entrar **diferenciada** (Δ); las I(0), en nivel (ver OE1).

In [ ]:
panel = pd.read_parquet(C.DATA_PROCESSED / 'panel.parquet')
print(panel.shape)
panel.head()

## 2. Baseline — cartera del sector (serie de tiempo, HAC)

In [ ]:
regresores = pm.FACTORES_GLOBALES + pm.FACTORES_LOCALES
regresores = [r for r in regresores if r in panel.columns]

cartera = panel['retorno'].groupby('fecha').mean()  # cartera equiponderada
Xmacro = panel[regresores].groupby('fecha').first()

res_base = pm.baseline_cartera(cartera, Xmacro, maxlags=4)
print(res_base.summary())

## 3. Modelo principal — Panel FE + Driscoll-Kraay

In [ ]:
regresores_panel = regresores + [c for c in pm.CONTROLES_EMPRESA if c in panel.columns]
res_fe = pm.panel_fe_driscoll_kraay(panel, dep='retorno', regresores=regresores_panel,
                                    entity_effects=True, time_effects=False)
print(res_fe)

## 4. Contraste H6 — Global vs Local

In [ ]:
tabla_h6 = pm.comparar_global_vs_local(res_fe)
display(tabla_h6)
tabla_h6.to_csv(C.TABLES / 'oe2_global_vs_local.csv')

## 5. Diagnósticos
- **CD-Pesaran**: si rechaza, hay dependencia de sección cruzada -> Driscoll-Kraay justificado.
- **VIF**: multicolinealidad entre regresores macro.
- **Hausman**: FE vs RE (en finanzas usualmente favorece FE).

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
Xv = sm.add_constant(panel[regresores_panel].dropna())
vif = pd.Series({Xv.columns[i]: variance_inflation_factor(Xv.values, i)
                 for i in range(Xv.shape[1])}, name='VIF')
vif

In [ ]:
# CD-Pesaran (test de dependencia de seccion cruzada sobre residuos del panel)
# Implementacion directa a partir de la matriz de residuos por empresa.
def cd_pesaran(residuos_wide: pd.DataFrame):
    R = residuos_wide.corr()
    N = R.shape[0]
    Tobs = residuos_wide.dropna().shape[0]
    rho_sum = (R.values[np.triu_indices(N, 1)]).sum()
    CD = np.sqrt(2*Tobs/(N*(N-1))) * rho_sum
    from scipy.stats import norm
    p = 2*(1-norm.cdf(abs(CD)))
    return CD, p

# Para usarlo: reconstruye los residuos del panel en formato wide (fecha x empresa)
# resid_wide = res_fe.resids.unstack(level='empresa')
# print('CD-Pesaran:', cd_pesaran(resid_wide))
print('Definida cd_pesaran(); aplicar sobre residuos del modelo FE.')

## 6. Lectura de resultados (OE2)
Documenta en prosa: signo y significancia de cada factor (¿se cumplen H1–H4?), y si el
bloque global domina al local (H6). Compara baseline (cartera) vs panel: si cuentan la
misma historia, la evidencia es robusta.

Próximo: `04_oe3_cointegracion.ipynb` (ARDL bounds / Johansen / VECM).